In [1]:
# @title 1. Настройки
MODELS_STORAGE = "Временная сессия Colab" # @param ["Временная сессия Colab", "Google Drive"]
OUTPUT_STORAGE = "Google Drive" # @param ["Временная сессия Colab", "Google Drive"]
DRIVE_MODELS_FOLDER = "AI_Models_for_Video" # @param {type:"string"}
DRIVE_OUTPUT_FOLDER = "ComfyUI_Output" # @param {type:"string"}
UPDATE_COMFY_UI = True # @param {type:"boolean"}

WORKSPACE = "/content/ComfyUI"

!nvidia-smi --query-gpu=name,memory.total --format=csv

name, memory.total [MiB]
NVIDIA A100-SXM4-80GB, 81920 MiB


In [2]:
# @title 2. Установка ComfyUI и зависимостей
import os

if not os.path.isdir(WORKSPACE):
    !git clone https://github.com/comfyanonymous/ComfyUI {WORKSPACE}
elif UPDATE_COMFY_UI:
    !git -C {WORKSPACE} pull

!pip install -q -r {WORKSPACE}/requirements.txt
!apt-get -y install -qq aria2
!command -v cloudflared >/dev/null || (wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb -O /tmp/cf.deb && dpkg -i /tmp/cf.deb)

print("✅ ComfyUI установлен")

Cloning into '/content/ComfyUI'...
remote: Enumerating objects: 65489, done.
remote: Counting objects: 100% (1095/1095), done.
remote: Compressing objects: 100% (405/405), done.
remote: Total 65489 (delta 922), reused 703 (delta 690), pack-reused 64394 (from 2)
Receiving objects: 100% (65489/65489), 100.13 MiB | 19.47 MiB/s, done.
Resolving deltas: 100% (45388/45388), done.
Selecting previously unselected package libcares2:amd64.
(Reading database ... 127324 files and directories currently installed.)
Preparing to unpack .../libcares2_1.27.0-1.0ubuntu1_amd64.deb ...
Unpacking libcares2:amd64 (1.27.0-1.0ubuntu1) ...
Selecting previously unselected package libaria2-0:amd64.
Preparing to unpack .../libaria2-0_1.37.0+debian-1build3_amd64.deb ...
Unpacking libaria2-0:amd64 (1.37.0+debian-1build3) ...
Selecting previously unselected package aria2.
Preparing to unpack .../aria2_1.37.0+debian-1build3_amd64.deb ...
Unpacking aria2 (1.37.0+debian-1build3) ...
Setting up libcares2:amd64 (1.27.0-1

In [3]:
# @title 3. Подключение хранилищ (модели и видео)
import os, shutil

SUBDIRS = ["checkpoints", "diffusion_models", "text_encoders", "vae",
           "loras", "clip_vision", "controlnet", "upscale_models"]

if "Google Drive" in (MODELS_STORAGE, OUTPUT_STORAGE) and not os.path.exists("/content/drive/MyDrive"):
    from google.colab import drive
    drive.mount('/content/drive')

# --- Модели ---
MODELS_LINK = f"{WORKSPACE}/models"
if MODELS_STORAGE == "Google Drive":
    MODELS_DIR = f"/content/drive/MyDrive/{DRIVE_MODELS_FOLDER}"
    for d in SUBDIRS:
        os.makedirs(f"{MODELS_DIR}/{d}", exist_ok=True)
    if os.path.islink(MODELS_LINK):
        os.unlink(MODELS_LINK)
    elif os.path.isdir(MODELS_LINK):
        shutil.rmtree(MODELS_LINK)
    os.symlink(MODELS_DIR, MODELS_LINK)
else:
    MODELS_DIR = MODELS_LINK
    if os.path.islink(MODELS_LINK):
        os.unlink(MODELS_LINK)
    for d in SUBDIRS:
        os.makedirs(f"{MODELS_DIR}/{d}", exist_ok=True)

# --- Готовые видео ---
if OUTPUT_STORAGE == "Google Drive":
    OUTPUT_DIR = f"/content/drive/MyDrive/{DRIVE_OUTPUT_FOLDER}"
else:
    OUTPUT_DIR = f"{WORKSPACE}/output"
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"📦 Модели: {MODELS_STORAGE} → {MODELS_DIR}")
print(f"🎬 Видео:  {OUTPUT_STORAGE} → {OUTPUT_DIR}")

Mounted at /content/drive
📦 Модели: Временная сессия Colab → /content/ComfyUI/models
🎬 Видео:  Google Drive → /content/drive/MyDrive/ComfyUI_Output


In [4]:
# @title 4. Скачивание моделей
import os
from google.colab import userdata

# @markdown Выбери, какую модель скачивать:
MODEL_CHOICE = "MiniMax H3"  # @param ["Wan 2.2", "MiniMax H3"]

try:
    CIVITAI_API_TOKEN = userdata.get('CIVITAI_API_TOKEN')
except Exception:
    CIVITAI_API_TOKEN = None

def downloadModel(url, subdir, filename=None):
    target_dir = f"{MODELS_DIR}/{subdir}"
    os.makedirs(target_dir, exist_ok=True)

    if 'huggingface.co' in url:
        full_url = url
        if filename is None:
            filename = url.split('/')[-1].split('?')[0]
    else:  # civitai
        full_url = f"{url}?token={CIVITAI_API_TOKEN}"

    if filename:
        path = f"{target_dir}/{filename}"
        if os.path.exists(path) and not os.path.exists(path + ".aria2"):
            print(f"✅ Уже есть: {filename}")
            return
        !aria2c --console-log-level=error -c -x 16 -s 16 -k 1M -d "{target_dir}" -o "{filename}" "{full_url}"
    else:
        !aria2c --console-log-level=error -c -x 16 -s 16 -k 1M -d "{target_dir}" "{full_url}"


# ---------- Списки моделей: (url, подпапка) ----------

WAN_HF = "https://huggingface.co/Comfy-Org/Wan-2.2/resolve/main/split_files"
WAN_22_MODELS = [
    (f"{WAN_HF}/diffusion_models/wan2.2_i2v_high_noise_14B_fp8_scaled.safetensors?download=true", "diffusion_models"),
    (f"{WAN_HF}/diffusion_models/wan2.2_i2v_low_noise_14B_fp8_scaled.safetensors?download=true", "diffusion_models"),
    (f"{WAN_HF}/text_encoders/umt5_xxl_fp8_e4m3fn_scaled.safetensors?download=true", "text_encoders"),
    (f"{WAN_HF}/vae/wan2.2_vae.safetensors?download=true", "vae"),
    # LightX2V LoRA
    (f"{WAN_HF}/loras/wan2.2_i2v_lightx2v_4steps_lora_v1_high_noise.safetensors?download=true", "loras"),
    (f"{WAN_HF}/loras/wan2.2_i2v_lightx2v_4steps_lora_v1_low_noise.safetensors?download=true", "loras"),
    (f"{WAN_HF}/vae/wan_2.1_vae.safetensors?download=true", "vae"),
]

MM_HF = "https://huggingface.co/Comfy-Org/MiniMax-H3/resolve/main"
MINIMAX_H3_MODELS = [
    (f"{MM_HF}/diffusion_models/minimax_h3_fl2va_pruned_int8_convrot.safetensors?download=true", "diffusion_models"),
    (f"{MM_HF}/vae/minimax_h3_video_vae_fp16.safetensors?download=true", "vae"),
    (f"{MM_HF}/text_encoders/qwen3vl_32b_minimax_h3_int8_convrot.safetensors?download=true", "text_encoders"),
    (f"{MM_HF}/vae/minimax_h3_audio_vae_fp32.safetensors?download=true", "vae"),
]

MODEL_SETS = {
    "Wan 2.2": WAN_22_MODELS,
    "MiniMax H3": MINIMAX_H3_MODELS,
}

# ---------- Скачивание выбранного набора ----------

print(f"⬇️ Выбрано: {MODEL_CHOICE}")
for url, subdir in MODEL_SETS[MODEL_CHOICE]:
    downloadModel(url, subdir)

!ls -lh {MODELS_DIR}/diffusion_models {MODELS_DIR}/text_encoders {MODELS_DIR}/vae
if MODEL_CHOICE == "Wan 2.2":
    !ls -lh {MODELS_DIR}/loras
print(f"🎉 Модели {MODEL_CHOICE} готовы")


Download Results:
gid   |stat|avg speed  |path/URI
======+====+===========+=======================================================
9b2410|OK  |   391MiB/s|/content/ComfyUI/models/diffusion_models/wan2.2_i2v_high_noise_14B_fp8_scaled.safetensors

Status Legend:
(OK):download completed.

Download Results:
gid   |stat|avg speed  |path/URI
======+====+===========+=======================================================
06a707|OK  |   408MiB/s|/content/ComfyUI/models/diffusion_models/wan2.2_i2v_low_noise_14B_fp8_scaled.safetensors

Status Legend:
(OK):download completed.

Download Results:
gid   |stat|avg speed  |path/URI
======+====+===========+=======================================================
edc390|OK  |   450MiB/s|/content/ComfyUI/models/text_encoders/umt5_xxl_fp8_e4m3fn_scaled.safetensors

Status Legend:
(OK):download completed.

Download Results:
gid   |stat|avg speed  |path/URI
======+====+===========+=======================================================
da5fee|OK  |   417MiB

In [ ]:
# @title 5. Запуск ComfyUI
TUNNEL = "cloudflared" # @param ["cloudflared", "localtunnel"]
EXTRA_FLAGS = "--enable-cors-header --enable-compress-response-body" # @param {type:"string"}

import subprocess, threading, time, socket, re, urllib.request

def tunnel_thread(port):
    while True:
        time.sleep(0.5)
        s = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
        ok = s.connect_ex(('127.0.0.1', port)) == 0
        s.close()
        if ok:
            break
    print(f"\nComfyUI загрузился, запускаю {TUNNEL}...\n")

    if TUNNEL == "cloudflared":
        p = subprocess.Popen(["cloudflared", "tunnel", "--protocol", "http2", "--edge-ip-version", "4",
                              "--url", f"http://127.0.0.1:{port}"],
                             stdout=subprocess.PIPE, stderr=subprocess.PIPE)
        for line in p.stderr:
            m = re.search(r"https://[-a-z0-9]+\.trycloudflare\.com", line.decode())
            if m:
                print("🔗 Ссылка на ComfyUI:", m.group(0))
    else:
        print("Пароль для localtunnel (IP):",
              urllib.request.urlopen('https://ipv4.icanhazip.com').read().decode().strip())
        p = subprocess.Popen(["lt", "--port", str(port)], stdout=subprocess.PIPE)
        for line in p.stdout:
            print(line.decode(), end='')

if TUNNEL == "localtunnel":
    !npm install -g localtunnel -q

threading.Thread(target=tunnel_thread, daemon=True, args=(8188,)).start()

%cd {WORKSPACE}
!python main.py --dont-print-server --output-directory "{OUTPUT_DIR}" {EXTRA_FLAGS}

/content/ComfyUI
[INFO] setup plugin alembic.autogenerate.schemas
[INFO] setup plugin alembic.autogenerate.tables
[INFO] setup plugin alembic.autogenerate.types
[INFO] setup plugin alembic.autogenerate.constraints
[INFO] setup plugin alembic.autogenerate.defaults
[INFO] setup plugin alembic.autogenerate.comments
[INFO] setup plugin alembic.ext.checkconstraint_byname
[INFO] Setting output directory to: /content/drive/MyDrive/ComfyUI_Output
[INFO] Found comfy_kitchen backend ascend: {'available': False, 'disabled': False, 'unavailable_reason': "torch-npu is not installed: No module named 'torch_npu'", 'capabilities': []}
[INFO] Found comfy_kitchen backend cuda: {'available': True, 'disabled': False, 'unavailable_reason': None, 'capabilities': ['adaln', 'apply_rope', 'apply_rope1', 'apply_rope1_', 'apply_rope_', 'apply_rope_split_half', 'apply_rope_split_half1', 'apply_rope_split_half1_', 'apply_rope_split_half_', 'closest_hit_on_mesh', 'closest_point_on_mesh', 'connected_components', 'co